# SFT Gemma 4 Audio + Unsloth + Verifiers + OpenFARM Bioacoustics

This notebook demonstrates supervised audio finetuning for OpenFARM bioacoustic tasks using Unsloth's Gemma 4 audio SFT path.

It follows the same Prime/Verifiers pattern as the OpenFARM RL notebook:

- install/load OpenFARM environments from the Prime environment hub;
- pull `get_dataset()` and `get_eval_dataset()` from the Verifiers environment interface;
- convert the environment prompts locally into Unsloth's Gemma 4 audio chat format;
- train with SFT;
- optionally score generated samples with the same Verifiers rubric used by Prime eval.

Default tasks:

- `soundwel_call_type`: pig vocalization HF/LF call type.
- `soundwel_valence`: pig vocalization positive/negative valence.
- `ungulate_valence`: cross-species ungulate contact-call valence.
- `laying_hen_stress_binary`: laying-hen stress-response binary label.

In [ ]:
%%capture
#@title Installation
import os, importlib.util
!pip install --upgrade -qqq uv

if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try:
        import numpy, PIL
        _numpy = f"numpy=={numpy.__version__}"
        _pil = f"pillow=={PIL.__version__}"
    except Exception:
        _numpy = "numpy"
        _pil = "pillow"

    # Gemma 4 requires transformers >= 5.5.0; do not pin transformers to 4.x.
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {_numpy} {_pil} torchvision bitsandbytes \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth

!uv pip install --upgrade --no-deps "transformers>=5.5.0" "tokenizers>=0.22.0,<=0.23.0" "trl>=0.28.0" unsloth unsloth_zoo
!uv pip install -qqq prime verifiers hatchling "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer wandb protobuf soundfile scipy nest_asyncio torchcodec
!uv pip install -qqq --no-deps --upgrade timm

import torch
torch._dynamo.config.recompile_limit = 64

## Configuration

The defaults are intentionally small enough for a Colab/Kaggle demonstration. Increase row limits and `max_steps` for a real run.

In [ ]:
#@title Model and run config
model_id = "unsloth/gemma-4-E2B-it" #@param ["unsloth/gemma-4-E2B-it", "unsloth/gemma-4-E4B-it"] {allow-input: true}
max_seq_length = 4096 #@param {type:"integer"}
lora_rank = 8 #@param {type:"integer"}
load_in_4bit = True #@param {type:"boolean"}
use_wandb = True #@param {type:"boolean"}
wandb_project = "openfarm-gemma4-audio-sft" #@param {type:"string"}
seed = 3301 #@param {type:"integer"}

#@title OpenFARM Verifiers environment config
team_id = "openfarm" #@param {type:"string"}
max_examples_per_task = 128 #@param {type:"integer"}
max_total_train_rows = 384 #@param {type:"integer"}
max_total_eval_rows = 96 #@param {type:"integer"}
train_split = "train" #@param {type:"string"}
test_split = "test" #@param {type:"string"}
balancing_strategy = "balanced" #@param ["balanced", "proportional"]
require_explanation = False #@param {type:"boolean"}
format_reward_weight = 0.2 #@param {type:"number"}
max_audio_seconds = 15.0 #@param {type:"number"}
audio_clip_strategy = "" #@param {type:"string"}
audio_sampling_rate = 16000 #@param {type:"integer"}

include_soundwel_call_type = True #@param {type:"boolean"}
include_soundwel_valence = True #@param {type:"boolean"}
include_ungulate_valence = True #@param {type:"boolean"}
include_laying_hen_stress_binary = True #@param {type:"boolean"}
include_catmeows_context = False #@param {type:"boolean"}

#@title SFT training/eval config
per_device_train_batch_size = 1 #@param {type:"integer"}
gradient_accumulation_steps = 4 #@param {type:"integer"}
learning_rate = 5e-5 #@param {type:"number"}
max_steps = 80 #@param {type:"integer"}
save_steps = 80 #@param {type:"integer"}
eval_steps_fraction = 0.25 #@param {type:"number"}
run_loss_eval_before_training = True #@param {type:"boolean"}
run_loss_eval_after_training = True #@param {type:"boolean"}
run_rubric_eval_before_training = True #@param {type:"boolean"}
run_rubric_eval_after_training = True #@param {type:"boolean"}
rubric_eval_examples = 8 #@param {type:"integer"}
rubric_eval_max_new_tokens = 128 #@param {type:"integer"}

In [ ]:
#@title Load Gemma 4 audio model with Unsloth
from unsloth import FastModel
import torch

model, processor = FastModel.from_pretrained(
    model_name = model_id,
    dtype = None,
    max_seq_length = max_seq_length,
    load_in_4bit = load_in_4bit,
    full_finetuning = False,
)

if use_wandb:
    import os
    import wandb
    os.environ["WANDB_PROJECT"] = wandb_project

## Build the OpenFARM EnvGroup

Each child task is loaded from the Prime environment hub with `modality="audio"`. The SFT notebook does not rebuild source-dataset logic; Verifiers remains the source of truth for prompts, splits, media clipping, answers, and rubrics.

In [ ]:
#@title Install and load OpenFARM bioacoustics environments
import json
import subprocess
from collections import Counter

import verifiers as vf
import verifiers.envs.env_group

# Keep EnvGroup routing Arrow-compatible when children already carry routes.
verifiers.envs.env_group._route_value = lambda route: list(route) if route else None


def hub_env(name: str) -> str:
    return f"{team_id}/{name}" if team_id else name


bio_tasks = []
if include_soundwel_call_type:
    bio_tasks.append(("bio_soundwel_call_type", "soundwel_call_type"))
if include_soundwel_valence:
    bio_tasks.append(("bio_soundwel_valence", "soundwel_valence"))
if include_ungulate_valence:
    bio_tasks.append(("bio_ungulate_valence", "ungulate_valence"))
if include_laying_hen_stress_binary:
    bio_tasks.append(("bio_hen_stress_binary", "laying_hen_stress_binary"))
if include_catmeows_context:
    bio_tasks.append(("bio_catmeows_context", "catmeows_context"))

if not bio_tasks:
    raise ValueError("Select at least one OpenFARM bioacoustic task.")

env_id = hub_env("openfarm-bioacustics")
subprocess.run(["prime", "env", "install", env_id], check = True)

env_specs = []
for name, task in bio_tasks:
    args = {
        "task": task,
        "modality": "audio",
        "train_split": train_split,
        "test_split": test_split,
        "max_examples_per_task": max_examples_per_task,
        "balancing_strategy": balancing_strategy,
        "require_explanation": require_explanation,
        "format_reward_weight": format_reward_weight,
    }
    if max_audio_seconds and max_audio_seconds > 0:
        args["max_audio_seconds"] = max_audio_seconds
    if audio_clip_strategy.strip():
        args["audio_clip_strategy"] = audio_clip_strategy.strip()
    env_specs.append({"name": name, "env_id": env_id, "args": args})

print("EnvGroup children:")
for spec in env_specs:
    print("-", spec["name"], spec["env_id"], spec["args"])

envs = [vf.load_environment(spec["env_id"], **spec["args"]) for spec in env_specs]
vf_env = vf.EnvGroup(envs = envs, env_names = [spec["name"] for spec in env_specs]) if len(envs) > 1 else envs[0]

keep_cols = ["prompt", "answer", "example_id", "info"]
train_env_dataset = vf_env.get_dataset()
train_env_dataset = train_env_dataset.select_columns([c for c in keep_cols if c in train_env_dataset.column_names])
train_env_dataset = train_env_dataset.shuffle(seed = seed)
if max_total_train_rows > 0:
    train_env_dataset = train_env_dataset.select(range(min(max_total_train_rows, len(train_env_dataset))))

eval_env_dataset = vf_env.get_eval_dataset()
eval_env_dataset = eval_env_dataset.select_columns([c for c in keep_cols if c in eval_env_dataset.column_names])
eval_env_dataset = eval_env_dataset.shuffle(seed = seed + 1)
if max_total_eval_rows > 0:
    eval_env_dataset = eval_env_dataset.select(range(min(max_total_eval_rows, len(eval_env_dataset))))


def maybe_json(value):
    return json.loads(value) if isinstance(value, str) else value


def route_name(info):
    info = maybe_json(info)
    route = info.get("env_id", "unknown") if isinstance(info, dict) else "unknown"
    if isinstance(route, list):
        return "/".join(str(part) for part in route)
    return str(route)

print("Train rows:", len(train_env_dataset))
print("Train route counts:", Counter(route_name(row["info"]) for row in train_env_dataset))
print("Eval rows:", len(eval_env_dataset))
print("Eval route counts:", Counter(route_name(row["info"]) for row in eval_env_dataset))
print("Sample env prompt:")
print(train_env_dataset[0]["prompt"])
print("Sample answer:", train_env_dataset[0]["answer"])
print("Sample info:", train_env_dataset[0]["info"])

## Convert Verifiers Prompts to Gemma 4 Audio SFT Messages

The environment already built the prompt and clipped/encoded media. This cell only adapts media parts from the Verifiers/OpenAI-style prompt representation into the Unsloth Gemma 4 audio chat format expected by `UnslothVisionDataCollator`.

In [ ]:
#@title Format Verifiers env examples for Unsloth SFT
import base64
import io
import math

import numpy as np
import soundfile as sf
from scipy.signal import resample_poly


def resample_audio_array(array, sr, target_sr):
    if not target_sr or int(sr) == int(target_sr):
        return array
    divisor = math.gcd(int(sr), int(target_sr))
    up = int(target_sr) // divisor
    down = int(sr) // divisor
    return resample_poly(array, up, down).astype(np.float32)


def decode_input_audio_part(part):
    audio_payload = part.get("input_audio") or {}
    data = audio_payload.get("data")
    if not data:
        raise ValueError("input_audio part is missing base64 data.")
    if "," in data and data.strip().startswith("data:"):
        data = data.split(",", 1)[1]
    audio_bytes = base64.b64decode(data)
    array, sr = sf.read(io.BytesIO(audio_bytes), dtype = "float32")
    array = np.asarray(array, dtype = np.float32)
    if array.ndim > 1:
        array = array.mean(axis = 1)
    return resample_audio_array(array, sr, audio_sampling_rate)


def normalize_content_to_unsloth(content):
    if isinstance(content, str):
        return [{"type": "text", "text": content}]

    converted = []
    for part in content:
        part_type = part.get("type")
        if part_type == "text":
            converted.append({"type": "text", "text": part.get("text", "")})
        elif part_type == "input_audio":
            converted.append({"type": "audio", "audio": decode_input_audio_part(part)})
        elif part_type == "audio":
            converted.append(part)
        else:
            raise ValueError(f"Unsupported prompt content part for audio SFT: {part_type!r}")
    return converted


def prompt_to_unsloth_messages(prompt):
    prompt = maybe_json(prompt)
    messages = []
    for turn in prompt:
        messages.append({
            "role": turn["role"],
            "content": normalize_content_to_unsloth(turn.get("content", "")),
        })
    return messages


def assistant_answer_message(answer):
    return {
        "role": "assistant",
        "content": [{"type": "text", "text": f"<answer>\n{answer}\n</answer>"}],
    }


def env_example_to_sft(example):
    messages = prompt_to_unsloth_messages(example["prompt"])
    messages.append(assistant_answer_message(example["answer"]))
    return {"messages": messages}


def compact_message_for_print(message):
    compact = []
    for turn in message:
        turn = {"role": turn["role"], "content": list(turn["content"])}
        for part in turn["content"]:
            if part.get("type") == "audio":
                part["audio"] = f"<audio array length={len(part['audio'])}>"
        compact.append(turn)
    return compact

train_dataset = train_env_dataset.map(
    env_example_to_sft,
    remove_columns = train_env_dataset.column_names,
    num_proc = 1,
)
eval_dataset = eval_env_dataset.map(
    env_example_to_sft,
    remove_columns = eval_env_dataset.column_names,
    num_proc = 1,
)

print("SFT train rows:", len(train_dataset))
print("SFT eval rows:", len(eval_dataset))
print("Sample SFT message:")
print(compact_message_for_print(train_dataset[0]["messages"]))

In [ ]:
#@title Quick baseline generation smoke test
from transformers import TextStreamer


def do_gemma_4_inference(messages, max_new_tokens = 128):
    was_training = model.training
    model.eval()
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt = True,
        tokenize = True,
        return_dict = True,
        return_tensors = "pt",
    ).to("cuda")
    with torch.no_grad():
        _ = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            do_sample = False,
            streamer = TextStreamer(processor, skip_prompt = True),
        )
    if was_training:
        model.train()

baseline_messages = train_dataset[0]["messages"][:-1]
do_gemma_4_inference(baseline_messages, max_new_tokens = 128)
print("Gold answer:", train_dataset[0]["messages"][-1]["content"][0]["text"])
print("Route:", route_name(train_env_dataset[0]["info"]))

## Train

Unsloth's audio SFT recipe uses `FastModel`, audio LoRA target modules, and `UnslothVisionDataCollator`, which handles multimodal token/audio alignment for Gemma 4.

In [ ]:
#@title Add LoRA adapters for Gemma 4 audio SFT
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,

    r = lora_rank,
    lora_alpha = lora_rank * 2,
    lora_dropout = 0,
    bias = "none",
    random_state = seed,
    use_rslora = False,
    loftq_config = None,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",

        # Audio layers used by the Unsloth Gemma 4 audio SFT template.
        "post", "linear_start", "linear_end",
        "embedding_projection",
        "ffw_layer_1", "ffw_layer_2",
        "output_proj",
    ],
)

In [ ]:
#@title SFT config
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTConfig, SFTTrainer

steps_between_evals = max(1, int(max_steps * eval_steps_fraction))

trainer = SFTTrainer(
    model = model,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    processing_class = processor.tokenizer,
    data_collator = UnslothVisionDataCollator(model, processor),
    args = SFTConfig(
        per_device_train_batch_size = per_device_train_batch_size,
        gradient_accumulation_steps = gradient_accumulation_steps,
        warmup_ratio = 0.03,
        max_steps = max_steps,
        learning_rate = learning_rate,
        logging_steps = 1,
        save_strategy = "steps",
        save_steps = save_steps,
        eval_strategy = "steps",
        eval_steps = steps_between_evals,
        per_device_eval_batch_size = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "cosine",
        seed = seed,
        output_dir = "outputs",
        report_to = "wandb" if use_wandb else "none",
        remove_unused_columns = False,

        # Required for multimodal/audio SFT examples that are already chat-formatted.
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = max_seq_length,
    ),
)

## Verifiers Rubric Eval Helper

This is not `vf_env.evaluate()` and does not need a model server. It generates locally with the Unsloth model, then scores completions with `vf_env.rubric.score_group(...)`, the same rubric contract used by Prime eval.

In [ ]:
#@title Local generation scored by the Verifiers rubric
import asyncio
import time
from collections import defaultdict

import nest_asyncio
from verifiers.utils.message_utils import normalize_messages
from verifiers.utils.thread_utils import get_or_create_thread_loop
from verifiers.utils.usage_utils import StateUsageTracker

nest_asyncio.apply()


def extract_generated_text(outputs, inputs):
    prompt_len = inputs["input_ids"].shape[-1]
    generated = outputs[0][prompt_len:]
    return processor.decode(generated, skip_special_tokens = True).strip()


def generate_for_env_example(example, max_new_tokens = rubric_eval_max_new_tokens):
    messages = prompt_to_unsloth_messages(example["prompt"])
    was_training = model.training
    model.eval()
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt = True,
        tokenize = True,
        return_dict = True,
        return_tensors = "pt",
    ).to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            do_sample = False,
        )
    if was_training:
        model.train()
    return extract_generated_text(outputs, inputs)


def build_scoring_state(example, completion_text, idx):
    info = maybe_json(example.get("info", {}))
    example_id = example.get("example_id") or f"local_eval_{idx}"
    prompt_msg = normalize_messages(maybe_json(example["prompt"]))
    comp_msg = normalize_messages([{"role": "assistant", "content": completion_text}])

    input_data = {
        "prompt": prompt_msg,
        "answer": example["answer"],
        "example_id": example_id,
        "info": info,
    }

    state = vf.State(input = input_data)
    state["prompt"] = prompt_msg
    state["completion"] = comp_msg
    state["trajectory_id"] = f"sft_eval_{idx}_{example_id}"
    state["trajectory"] = [{
        "prompt": prompt_msg,
        "completion": comp_msg,
        "response": None,
        "tokens": None,
        "reward": None,
        "advantage": None,
        "is_truncated": False,
        "trajectory_id": state["trajectory_id"],
        "extras": {},
    }]
    state["task"] = info.get("env_id", "openfarm") if isinstance(info, dict) else "openfarm"
    state["metrics"] = {}
    state["error"] = None
    state["is_completed"] = True
    state["is_truncated"] = False
    state["client"] = None
    state["model"] = "unsloth-gemma4-audio-sft"
    state["usage_tracker"] = StateUsageTracker()
    state["tool_defs"] = []
    state["timing"] = {
        "generation_ms": 0.0,
        "scoring_ms": 0.0,
        "total_ms": 0.0,
        "start_time": time.time(),
    }
    return state


def run_verifiers_rubric_eval(dataset, n = rubric_eval_examples, label = "eval"):
    if n <= 0:
        print(f"Skipping {label} Verifiers rubric eval because n <= 0.")
        return {}

    n = min(n, len(dataset))
    states = []
    rows = []
    for idx in range(n):
        example = dataset[idx]
        completion = generate_for_env_example(example)
        rows.append({
            "idx": idx,
            "route": route_name(example["info"]),
            "gold": example["answer"],
            "completion": completion,
        })
        states.append(build_scoring_state(example, completion, idx))

    groups = defaultdict(list)
    for state in states:
        groups[state["input"]["example_id"]].append(state)

    loop = get_or_create_thread_loop()

    async def score_and_cleanup():
        await asyncio.gather(*(vf_env.rubric.score_group(g) for g in groups.values()))
        if hasattr(vf_env.rubric, "cleanup"):
            await asyncio.gather(*(vf_env.rubric.cleanup(s) for s in states))

    loop.run_until_complete(score_and_cleanup())

    rewards = [float(state.get("reward") or 0.0) for state in states]
    metric_values = defaultdict(list)
    for state in states:
        for key, value in state.get("metrics", {}).items():
            try:
                metric_values[key].append(float(value))
            except (TypeError, ValueError):
                pass

    summary = {f"{label}/rubric_reward": sum(rewards) / len(rewards) if rewards else 0.0}
    for key, values in metric_values.items():
        summary[f"{label}/rubric_{key}"] = sum(values) / len(values)

    print(f"{label} Verifiers rubric summary:", summary)
    print("Sample scored generations:")
    for row, reward in zip(rows[:3], rewards[:3]):
        print({**row, "reward": reward})

    if use_wandb:
        import wandb
        if wandb.run is not None:
            wandb.log(summary)
    return summary

In [ ]:
#@title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
#@title Run SFT training
if run_loss_eval_before_training:
    print("Running baseline loss eval before SFT...")
    baseline_loss_metrics = trainer.evaluate(metric_key_prefix = "eval_baseline")
    print("Baseline loss eval:", baseline_loss_metrics)

if run_rubric_eval_before_training:
    print("Running baseline Verifiers rubric eval before SFT...")
    baseline_rubric_metrics = run_verifiers_rubric_eval(eval_env_dataset, label = "eval_baseline")

trainer_stats = trainer.train()

if run_loss_eval_after_training:
    print("Running final loss eval after SFT...")
    final_loss_metrics = trainer.evaluate(metric_key_prefix = "eval_final")
    print("Final loss eval:", final_loss_metrics)

if run_rubric_eval_after_training:
    print("Running final Verifiers rubric eval after SFT...")
    final_rubric_metrics = run_verifiers_rubric_eval(eval_env_dataset, label = "eval_final")

In [ ]:
#@title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

## Inference Smoke Test

In [ ]:
#@title Generate after SFT
sample_messages = eval_dataset[0]["messages"]
do_gemma_4_inference(sample_messages[:-1], max_new_tokens = 128)
print("Gold answer:", sample_messages[-1]["content"][0]["text"])
print("Route:", route_name(eval_env_dataset[0]["info"]))

## Save

In [ ]:
#@title Save LoRA locally and optionally push to Hugging Face
output_lora_dir = "openfarm_gemma4_audio_sft_lora" #@param {type:"string"}
push_to_hub = False #@param {type:"boolean"}
hf_lora_repo = "YOUR_USERNAME/openfarm-gemma4-audio-sft-lora" #@param {type:"string"}

model.save_pretrained(output_lora_dir)
processor.save_pretrained(output_lora_dir)
print("Saved LoRA to", output_lora_dir)

if push_to_hub:
    import os
    model.push_to_hub(hf_lora_repo, token = os.environ.get("HF_TOKEN"))
    processor.push_to_hub(hf_lora_repo, token = os.environ.get("HF_TOKEN"))
    print("Pushed LoRA to", hf_lora_repo)

## Suggested video caption

"OpenFARM environments are not only benchmarks. They are trainable research tasks: here, Gemma 4 learns directly from Prime/Verifiers audio environments, with Unsloth SFT handling low-cost adaptation and the same Verifiers rubrics checking whether model outputs improve."